# ⚙️ Notebook 3: Feature Engineering

# 03 Feature Engineering
This notebook applies the core feature engineering pipeline  including time-series specific features like lags, rolling statistics, and holiday indicators.

In [ ]:
import pandas as pd
import numpy as np
import holidays
import os

# Load cleaned data
df = pd.read_csv("../data/cleaned_data.csv", parse_dates=['Date'])

## 1. Time-Based Features
Extracting temporal components that capture seasonality.

In [ ]:
def add_time_features(df):
    df = df.copy()
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
    df['Quarter'] = df['Date'].dt.quarter
    return df

df = add_time_features(df)
df.head()

## 2. Holiday Indicators
Flagging major US and Indian holidays to account for sales spikes.

### Impact of Holidays on Sales
Holiday flags are binary indicators (0 or 1) that tell the model if a specific week contains a major holiday. **Holidays fundamentally shift consumer buying behavior**, often resulting in significant sales spikes (e.g., Diwali, Christmas, or Black Friday). Without these flags, the model would view these spikes as random errors; with them, the model learns to expect and predict higher volume during these critical periods.

In [ ]:
us_holidays = holidays.US()
india_holidays = holidays.India()

df['Is_US_Holiday'] = df['Date'].apply(lambda x: 1 if x in us_holidays else 0)
df['Is_Indian_Holiday'] = df['Date'].apply(lambda x: 1 if x in india_holidays else 0)
df[['Date', 'Is_US_Holiday', 'Is_Indian_Holiday']].head()

## 3. Lags and Rolling Windows
Incorporating historical performance as input features (autocorrelation).

### Why Lag Features Help
Lag features are essentially the sales from previous time periods (e.g., last week or last month) shifted forward. They are critical because **past sales are often the strongest predictors of future sales**—a concept known as autocorrelation. By providing the model with these historical values, we enable it to learn momentum, cyclic patterns, and the baseline volume for each state.

### The Power of Rolling Statistics
Rolling statistics (like a 4-week mean) help **smooth out short-term noise and random fluctuations** in the data. While individual weeks might have spikes due to minor local events, the rolling mean provides a clearer picture of the underlying trend direction, helping the model focus on stable patterns rather than getting distracted by data 'noise'.

In [ ]:
def add_history_features(df):
    df = df.sort_values(['State', 'Date'])
    
    # Lags
    df['Lag_1'] = df.groupby('State')['Total'].shift(1)
    df['Lag_7'] = df.groupby('State')['Total'].shift(7)
    df['Lag_30'] = df.groupby('State')['Total'].shift(30)
    
    # Rolling Stats
    df['Rolling_Mean_4'] = df.groupby('State')['Total'].transform(lambda x: x.shift(1).rolling(window=4).mean())
    df['Rolling_Std_4'] = df.groupby('State')['Total'].transform(lambda x: x.shift(1).rolling(window=4).std())
    
    return df

df = add_history_features(df)
df = df.dropna()
print(f"Final feature set shape: {df.shape}")
df.head()

## 4. Save Engineered Dataset

In [ ]:
df.to_csv("../data/features_data_v2.csv", index=False)
print("Engineered features saved to ../data/features_data_v2.csv")